# Auditing a model on `adult`

A full isitfair audit of a logistic regression fitted to the public
`adult` dataset: discrimination, calibration, decision curves,
Hosmer-Lemeshow, estimability, a recalibration ladder, mitigation and an
HTML report.

Before running, build the prediction table once from the repository root:

    python -I extra_examples/datasets.py adult

`B = 50` bootstrap replicates keep the run short; use 2,000 for results you
intend to report.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

for _c in (Path.cwd(), *Path.cwd().parents):
    if (_c / "extra_examples" / "datasets.py").exists():
        HERE = _c / "extra_examples"
        break
sys.path.insert(0, str(HERE))
import datasets  # noqa: E402

import isitfair  # noqa: E402
from isitfair import FairnessAudit, GroupRecalibration, WassersteinPostprocessing  # noqa: E402

NAME = "adult"
B = 50
SEED = 42
META = datasets.meta(NAME)
THRESHOLD = META["threshold"]
ATTRS = datasets.ATTRS[NAME]
print(f"isitfair {isitfair.__version__}")
print(f"{NAME}: outcome {META['outcome']}; threshold {THRESHOLD}; B = {B}")

isitfair 0.4.0
adult: outcome income > 50K (the test file's labels carry a trailing period); threshold 0.3; B = 50


## 1. Load the predictions

One row per test-set patient or person: the outcome, the model's predicted
probability, and the attributes to audit.

In [2]:
df = datasets.load(NAME)
print(f"n = {len(df)}, prevalence = {df['y_true'].mean():.3f}")
for a in ATTRS:
    print(f"{a}: {df[a].nunique()} levels, {int(df[a].isna().sum())} missing")

n = 14653, prevalence = 0.239
sex: 2 levels, 0 missing
race: 5 levels, 0 missing
age_group: 4 levels, 0 missing


isitfair refuses missing subgroup labels rather than silently dropping those
rows. If any attribute has missing labels, this cell shows the error and then
recodes them to an explicit `"Unknown"` level.

In [3]:
missing_attrs = [a for a in ATTRS if df[a].isna().any()]
if missing_attrs:
    try:
        FairnessAudit(y_true=df["y_true"], y_proba=df["y_proba"],
                      sensitive_features={a: df[a] for a in ATTRS},
                      threshold=THRESHOLD, n_bootstrap=B, random_state=SEED)
    except ValueError as exc:
        print("ValueError:", exc)
    for a in missing_attrs:
        df[a] = df[a].fillna("Unknown")
print("recoded to 'Unknown':", missing_attrs or "none")

recoded to 'Unknown': none


## 2. Split off a calibration half for mitigation

The test half is split 50/50, stratified on the outcome. Mitigation
calibrators are fitted on one half and every audit runs on the other, so no
row is scored by a calibrator fitted on it.

In [4]:
i_cal, i_eval = train_test_split(np.arange(len(df)), test_size=0.5, random_state=SEED,
                                 stratify=df["y_true"])
cal_df, eval_df = df.iloc[i_cal].reset_index(drop=True), df.iloc[i_eval].reset_index(drop=True)
# mitigate() rejects audited levels that the calibration half never saw; drop
# such rows (for example diabetes130's single "Unknown/Invalid" gender).
unseen = np.zeros(len(eval_df), bool)
for a in ATTRS:
    unseen |= ~eval_df[a].isin(set(cal_df[a]))
if unseen.any():
    print(f"dropping {int(unseen.sum())} audited row(s) with levels absent from the calibration half")
    eval_df = eval_df[~unseen].reset_index(drop=True)
print(f"calibration n = {len(cal_df)}, audit n = {len(eval_df)}")

calibration n = 7326, audit n = 7327


## 3. Construct the audit

Each attribute is audited on its own, and the first two are also crossed
(passing `None` for a tuple key asks isitfair to build the intersection).

In [5]:
y = eval_df["y_true"].to_numpy(dtype=float)
p = eval_df["y_proba"].to_numpy(dtype=float)
marginals = {a: eval_df[a].to_numpy(dtype=object) for a in ATTRS}
INTERSECTION = tuple(ATTRS[:2]) if len(ATTRS) >= 2 else None
sf = dict(marginals)
if INTERSECTION:
    sf[INTERSECTION] = None

audit = FairnessAudit(y_true=y, y_proba=p, sensitive_features=sf, threshold=THRESHOLD,
                      n_bootstrap=B, random_state=SEED)
audit

FairnessAudit(n=7327, events=1753, threshold=0.3, n_bootstrap=50, random_state=42, attributes: sex (2), race (5), age_group (4), sex × race (10))

## 4. Discrimination

In [6]:
disc = audit.discrimination()
disc[disc.subgroup != "GAP"][["attribute", "subgroup", "n", "n_events", "auroc",
                              "auroc_ci_low", "auroc_ci_high"]]

,attribute,subgroup,n,n_events,auroc,auroc_ci_low,auroc_ci_high
0,sex,Overall,7327.0,1753.0,0.899383,0.888671,0.906093
1,sex,Female,2456.0,257.0,0.925976,0.914633,0.940679
2,sex,Male,4871.0,1496.0,0.872684,0.864268,0.881473
7,race,Overall,7327.0,1753.0,0.899383,0.888671,0.906093
8,race,Amer-Indian-Eskimo,81.0,10.0,0.933803,NaN,NaN
9,race,Asian-Pac-Islander,211.0,62.0,0.888612,0.838795,0.919940
10,race,Black,697.0,89.0,0.937001,0.917623,0.956991
11,race,Other,63.0,6.0,0.885965,NaN,NaN
12,race,White,6275.0,1586.0,0.894053,0.886923,0.899902
17,age_group,Overall,7327.0,1753.0,0.899383,0.888671,0.906093


## 5. Calibration

In [7]:
cal = audit.calibration()
cal[cal.subgroup != "GAP"][["attribute", "subgroup", "n", "calibration_intercept",
                            "calibration_slope", "brier", "ece", "ici"]]

,attribute,subgroup,n,calibration_intercept,calibration_slope,brier,ece,ici
0,sex,Overall,7327.0,-0.000782,0.937857,0.106113,0.013623,0.010995
1,sex,Female,2456.0,-0.224076,1.036449,0.054831,0.013080,0.012864
2,sex,Male,4871.0,0.052646,0.878971,0.131970,0.022542,0.015266
5,race,Overall,7327.0,-0.000782,0.937857,0.106113,0.013623,0.010995
6,race,Amer-Indian-Eskimo,81.0,-0.649103,1.155874,0.061509,0.062194,0.048540
7,race,Asian-Pac-Islander,211.0,0.176366,0.822235,0.126835,0.070126,0.051209
8,race,Black,697.0,-0.136132,1.165103,0.060373,0.015667,0.012601
9,race,Other,63.0,-0.513724,0.513002,0.072278,0.074447,0.053939
10,race,White,6275.0,0.011061,0.925271,0.111412,0.015148,0.010809
13,age_group,Overall,7327.0,-0.000782,0.937857,0.106113,0.013623,0.010995


In [8]:
curves = audit.calibration_curves()
roc = audit.roc_curves()
print("calibration curves:", curves.shape, "| ROC curves:", roc.shape)

calibration curves: (2500, 7) | ROC curves: (2500, 7)


## 6. Decision curve: gaps in net benefit between subgroups

In [9]:
dc = audit.decision_curve()
dc[dc.subgroup == "GAP"][["attribute", "gap_value", "gap_ci_low", "gap_ci_high"]]

,attribute,gap_value,gap_ci_low,gap_ci_high
60,sex,0.202952,0.187635,0.220808
181,race,0.203766,NaN,NaN
282,age_group,0.321155,0.293936,0.337869
503,sex × race,0.341872,NaN,NaN


## 7. Estimability

Cells with too few events for a metric are flagged rather than reported.

In [10]:
est = audit.estimability()
print(f"{int(est.not_estimable_final.sum())} of {len(est)} cell-metrics not reportable")

98 of 303 cell-metrics not reportable


## 8. Hosmer-Lemeshow

In [11]:
hl = audit.hosmer_lemeshow()
hl[hl.subgroup != "GAP"][["attribute", "subgroup", "n", "hl_statistic", "hl_df",
                          "hl_p_value", "n_bins_used"]]

,attribute,subgroup,n,hl_statistic,hl_df,hl_p_value,n_bins_used
0,sex,Overall,7327.0,25.150479,10.0,5.067456e-03,10.0
1,sex,Female,2456.0,11.568700,10.0,3.149579e-01,10.0
2,sex,Male,4871.0,37.258866,10.0,5.104443e-05,10.0
4,race,Overall,7327.0,25.150479,10.0,5.067456e-03,10.0
5,race,Amer-Indian-Eskimo,81.0,4.704887,10.0,9.100003e-01,10.0
6,race,Asian-Pac-Islander,211.0,13.284066,10.0,2.082202e-01,10.0
7,race,Black,697.0,5.252320,10.0,8.736979e-01,10.0
8,race,Other,63.0,8.996513,10.0,5.324346e-01,10.0
9,race,White,6275.0,29.891585,10.0,8.923236e-04,10.0
11,age_group,Overall,7327.0,25.150479,10.0,5.067456e-03,10.0


## 9. Intersection cells

In [12]:
audit.cell_diagnostics() if INTERSECTION else "single attribute; no intersection"

,attribute,subgroup,n,n_events,shrunk,marginal_rate,cell_rate,shrinkage_factor
0,sex × race,Female / Amer-Indian-Eskimo,33,4,True,0.104642,0.121212,0.948960
1,sex × race,Female / Asian-Pac-Islander,66,7,True,0.104642,0.106061,0.902877
2,sex × race,Female / Black,350,22,True,0.104642,0.062857,0.636760
3,sex × race,Female / Other,20,1,True,0.104642,0.050000,0.968432
4,sex × race,Female / White,1987,223,False,0.104642,0.112229,0.235931
5,sex × race,Male / Amer-Indian-Eskimo,48,6,True,0.307124,0.125000,0.927443
6,sex × race,Male / Asian-Pac-Islander,145,55,False,0.307124,0.379310,0.808846
7,sex × race,Male / Black,347,67,False,0.307124,0.193084,0.638749
8,sex × race,Male / Other,43,5,True,0.307124,0.116279,0.934506
9,sex × race,Male / White,4288,1363,False,0.307124,0.317864,0.125175


## 10. Recalibration ladder on the attribute with the largest intercept gap

A level with no events at all cannot be recalibrated, and isitfair says so.
In that case the ladder moves on to the next attribute.

In [13]:
gaps = cal[(cal.subgroup == "GAP") & (cal.gap_metric == "max_intercept_gap")
           & cal.attribute.isin(ATTRS)]
RANKED = [str(a) for a in gaps.sort_values("gap_value", ascending=False).attribute]
ladder = None
for by in RANKED:
    try:
        ladder = audit.recalibration_ladder(by=by)
        print("ladder by", by)
        break
    except ValueError as exc:
        print(f"{by}: {exc}")
ladder

ladder by age_group


,rung,attribute,subgroup,n,n_events,calibration_intercept,intercept_ci_low,intercept_ci_high,calibration_slope,slope_ci_low,...,brier_ci_low,brier_ci_high,ece,ici,net_benefit,gap_metric,gap_value,event_floor,below_floor,not_estimable_final
0,none,age_group,Overall,7327.0,1753.0,-0.000782,-0.049989,0.068019,0.937857,0.899830,...,0.102988,0.110385,0.013623,0.010995,0.130574,NaN,NaN,25.0,False,False
1,none,age_group,17-29,2225.0,116.0,-0.658289,-0.844317,-0.489810,1.188757,1.039773,...,0.028933,0.036513,0.026418,0.025653,0.014703,NaN,NaN,25.0,False,False
2,none,age_group,30-44,2725.0,814.0,0.149713,0.010232,0.272203,0.904165,0.851816,...,0.124707,0.137638,0.025173,0.021097,0.162097,NaN,NaN,25.0,False,False
3,none,age_group,45-59,1746.0,659.0,0.233292,0.113623,0.355173,0.901687,0.788419,...,0.135805,0.156076,0.034897,0.029689,0.231304,NaN,NaN,25.0,False,False
4,none,age_group,60+,631.0,164.0,-0.629405,-0.819804,-0.409497,0.706176,0.569786,...,0.128865,0.152447,0.077444,0.072704,0.124293,NaN,NaN,25.0,False,False
5,none,age_group,GAP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,max_intercept_gap,0.891582,NaN,NaN,NaN
6,common,age_group,Overall,7327.0,1753.0,0.000598,-0.047346,0.065908,0.902606,0.841188,...,0.103210,0.110786,0.007242,0.003205,0.133108,NaN,NaN,25.0,False,False
7,common,age_group,17-29,2225.0,116.0,-0.646475,-0.830584,-0.484251,1.002413,0.684495,...,0.029500,0.037069,0.025865,0.026127,0.016308,NaN,NaN,25.0,False,False
8,common,age_group,30-44,2725.0,814.0,0.136349,0.000771,0.258674,0.865157,0.780864,...,0.125022,0.137148,0.021035,0.016443,0.164089,NaN,NaN,25.0,False,False
9,common,age_group,45-59,1746.0,659.0,0.229951,0.113797,0.349608,0.928854,0.786507,...,0.137700,0.157636,0.035234,0.031919,0.234904,NaN,NaN,25.0,False,False


## 11. Mitigation

Group-wise isotonic recalibration, then Wasserstein post-processing, both
fitted on the calibration half.

In [14]:
calib_data = (cal_df["y_true"].to_numpy(float), cal_df["y_proba"].to_numpy(float),
              {a: cal_df[a].to_numpy(object) for a in ATTRS})
mitigated_gr = audit.mitigate(method=GroupRecalibration(method="isotonic"), calibration_data=calib_data)
a0 = ATTRS[0]
keep = ["subgroup", "calibration_intercept", "calibration_slope", "brier", "ece"]
before, after = audit.calibration(), mitigated_gr[a0].calibration()
print("Before,", a0)
print(before[before.attribute == a0][keep].to_string(index=False))
print("\nAfter group isotonic recalibration,", a0)
print(after[after.attribute == a0][keep].to_string(index=False))

Before, sex
subgroup  calibration_intercept  calibration_slope    brier      ece
 Overall              -0.000782           0.937857 0.106113 0.013623
  Female              -0.224076           1.036449 0.054831 0.013080
    Male               0.052646           0.878971 0.131970 0.022542
     GAP                    NaN                NaN      NaN      NaN
     GAP                    NaN                NaN      NaN      NaN

After group isotonic recalibration, sex
subgroup  calibration_intercept  calibration_slope    brier      ece
 Overall              -0.022697           0.862836 0.105881 0.013011
  Female              -0.150329           0.798642 0.054789 0.010140
    Male               0.004849           0.861158 0.131643 0.019134
     GAP                    NaN                NaN      NaN      NaN
     GAP                    NaN                NaN      NaN      NaN


In [15]:
mitigated_w = audit.mitigate(method=WassersteinPostprocessing(random_state=SEED), calibration_data=calib_data)
d_after = mitigated_w[a0].discrimination()
d_after[(d_after.attribute == a0) & (d_after.subgroup == "GAP")][
    ["gap_metric", "gap_value", "gap_ci_low", "gap_ci_high"]]

,gap_metric,gap_value,gap_ci_low,gap_ci_high
3,equal_opportunity_difference,0.197081,0.152761,0.243395
4,predictive_equality,0.068293,0.049927,0.084129
5,equalized_odds_gap,0.197081,0.152761,0.243395
6,statistical_parity,0.029717,0.012458,0.049079


## 12. HTML report

In [16]:
REPORT = HERE / "reports" / f"{NAME}.html"
REPORT.parent.mkdir(exist_ok=True)
audit.report(out=REPORT, title=f"{NAME}: isitfair audit",
             metadata={"Dataset": NAME, "Model": "LogisticRegression",
                       "Threshold": str(THRESHOLD), "Bootstrap": f"B={B}, random_state={SEED}"},
             mitigation_audits=mitigated_gr)
print("wrote", REPORT.relative_to(HERE.parent))

wrote extra_examples\reports\adult.html
